In [5]:
!pip install pymorphy3 nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 56.6 MB/s eta 0:00:00


In [8]:
import re
import string
from collections import Counter
from typing import List, Tuple


import pymorphy3
import nltk
from nltk.corpus import stopwords


nltk.download('stopwords')
nltk.download('punkt')
russian_stopwords = set(stopwords.words('russian'))


extra_stopwords = {'это', 'так', 'все', 'уже', 'ещё', 'также', 'быть', 'мочь',
                   'сказать', 'знать', 'стать', 'говорить', 'пойти', 'стать',
                   'иметь', 'являться', 'находиться', 'составить', 'сообщить',
                   'отметить', 'подчеркнуть', 'показать', 'стать', 'начать',
                   'продолжить', 'стать', 'заявить', 'рассказать', 'получить'}

russian_stopwords.update(extra_stopwords)

morph = pymorphy3.MorphAnalyzer()

def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s\.\!\?]', '', text)
    return text.strip()

def lemmatize_word(word: str) -> str:
    return morph.parse(word)[0].normal_form

def preprocess_text(text: str) -> List[str]:
    words = re.findall(r'\b\w+\b', text.lower())
    words = [w for w in words if len(w) >= 3 and not w.isdigit()]
    lemmas = [lemmatize_word(w) for w in words]
    lemmas = [l for l in lemmas if l not in russian_stopwords]
    return lemmas

def split_sentences(text: str) -> List[str]:
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]
    return sentences

def compute_word_frequencies(lemmas_list: List[str]) -> Counter:
    return Counter(lemmas_list)

def score_sentences(text: str, use_average: bool = False, top_k: int = 4) -> List[str]:
    full_lemmas = preprocess_text(text)
    freq = compute_word_frequencies(full_lemmas)

    sentences = split_sentences(text)
    if not sentences:
        return []

    scored = []
    for sent in sentences:
        sent_lemmas = preprocess_text(sent)
        if not sent_lemmas:
            continue
        total_score = sum(freq.get(word, 0) for word in sent_lemmas)
        if use_average:
            avg_score = total_score / len(sent_lemmas) if sent_lemmas else 0
            scored.append((avg_score, sent))
        else:
            scored.append((total_score, sent))

    scored.sort(key=lambda x: x[0], reverse=True)

    if use_average:
        return [scored[0][1]]
    else:
        return [sent for _, sent in scored[:top_k]]



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [14]:
article_apple = """
Apple отчиталась о рекордном спросе на линейку iPhone 17, а рекордные продажи цифровых услуг помогли технологическому гиганту превзойти ожидания аналитиков. За минувший квартал прибыль компании составила $29,6 млрд при выручке в $111,2 млрд.
Как сообщает AFP, генеральный директор Тим Кук назвал этот период «лучшим мартовским кварталом за всю историю». По его словам, продажи смартфонов продемонстрировали двузначный рост почти во всех регионах присутствия.
20 апреля Тим Кук заявил, что покинет пост генерального директора и станет исполнительным председателем совета директоров компании. Его преемником назначен Джон Тернус, который сейчас занимает пост вице-президента по аппаратной инженерии.
"""

article_autos = """
Количество шоурумов автомобильных брендов в России снова начало расти: к концу марта общая сеть достигла 4,2 тыс. дилерских центров. Разогревают интерес к открытию салонов новые суббренды крупных китайских производителей, а также собирающиеся в России и Белоруссии марки. Дилеры в оценках осторожны: они говорят не столько о росте сети, сколько о трансформации рынка в целом.
Количество шоурумов легковых и легких коммерческих автомобилей в России вновь вернулось к росту после массовой коррекции в 2025 году, следует из исследования «Газпромбанк Автолизинга» и «Автобизнесревю». По итогам первого квартала 2026 года дилерская сеть достигла 4,2 тыс. точек. Это на 2,6% больше, чем в первом квартале 2025 года, и на 2,8% превышает уровень начала года.
Число автосалонов китайских марок увеличилось на 3,2% относительно начала года, до 2,7 тыс. штук, после снижения в 2025 году (см. “Ъ” от 21 января). По данным аналитиков, состоялись 154 открытия и 70 закрытий. Количество шоурумов брендов из КНР осталось примерно на уровне первого квартала 2025 года, доля — 64%.
Рост дилерских площадок в первом квартале — перестройка рынка, считает президент ассоциации «Российские автомобильные дилеры» Алексей Подщеколдин. «Конечно, есть и открытие новых площадок как у новых брендов, так и запуск новых дилерских центров под уже знакомыми марками. Это нормальный процесс. Но в большей степени мы сейчас видим именно смену вывесок»,— рассказывает он.
Во многом речь идет не о появлении принципиально новых шоурумов, вторит директор департамента продаж новых автомобилей компании «Рольф» Николай Иванов, а о расширении дилерских контрактов за счет выхода на рынок новых брендов и суббрендов, прежде всего со стороны китайских производителей. В ряде случаев на базе уже существующей инфраструктуры подписываются дополнительные дилерские соглашения, что статистически увеличивает количество «площадок», но не всегда отражает фактическое расширение сети. В китайском сегменте, уточняет топ-менеджер, один и тот же дилер может работать сразу с несколькими брендами внутри одной группы, что формально дает рост по количеству контрактов.
Директор по маркетингу АГ «Авилон» Юрий Блинов добавляет, что растет не дилерская сеть в целом, а сеть у тех, кто пережил фазу рыночной эйфории и вышел в фазу экономики.
«Для рынка это хорошо. Потому что нормальный шоурум открывают не ради красивой вывески, а ради продаж, сервиса и оборота. Все остальное рынок уже расставил на места»,— говорит он.
В целом динамика — отскок после коррекции 2025 года, полагает топ-менеджер: рынок не разгоняется заново, а пересобирается в более рациональную конфигурацию.
В лидерах по числу открытий в первом квартале — новичок рынка суббренд китайского Changan — Deepal (плюс 65 точек). Остальные марки из КНР корректировали число шоурумов в незначительных по сравнению с прежними квартальными периодами масштабах. В этих условиях, уточняется в исследовании, наиболее заметным оказался рост сетей российского Evolutе — на 17 новых точек, белорусского Avior (де-факто китайский SAIC Motor) — на 14, корейской KGM — на 9 точек. Динамику Evolute эксперты объясняют участием гибридного кроссовера марки в программе льготного кредитования и положительной реакцией рынка на снижение цен. «Дилеры быстро увидели потенциал, и сеть за квартал увеличилась практически в полтора раза, до 50 салонов»,— уточняется в исследовании.
Самая масштабная дилерская сеть (218 локаций) среди китайских брендов остается у Chery, планирующей сокращать присутствие в стране (см. “Ъ” от 17 сентября 2025 года). Все точки заполнены по большей части российско-китайскими автомобилями Tenet. Следом по количеству шоурумов идут Geely (210 точек) и Changan (201 точка).
Во втором квартале, вероятно, активнее всего продолжат расширять свои сети не китайские бренды, а Evolutе, Avior и KGM, отмечается в исследовании. Руководитель управления по работе с импортерами «Газпромбанк Автолизинга» Александр Корнев ожидает, что во второй половине года лидером по открытиям может стать бренд Volga, дистрибутор которого объявил о поиске 130 партнеров, и дилеры активно раскупают франшизы. Кроме того, прогнозирует топ-менеджер, продолжится переформатирование салонов суббрендов Chery под новые марки с российским капиталом. «В частности, кроссоверы, продававшиеся у нас ранее под брендом Jaecoo, уже начали собирать на заводе в Санкт-Петербурге под маркой Jeland»,— напоминает он.
"""

article_palladium = """
Вопреки ожиданиям конфликт на Ближнем Востоке оказывает на палладий понижательное давление: фьючерсы опустились ниже $1,4 тыс. за унцию из-за укрепления доллара, фиксации прибыли и высоких ставок.
В настоящий момент рынок палладия фиксирует снижение цен, однако на среднесрочном горизонте создаются предпосылки для дефицита металла.
США ввели предварительную компенсационную пошлину в размере 109,1% на российский палладий, а Евросоюз в рамках принятого 20-го пакета санкций может запретить его импорт.
Эти меры фактически блокируют западные рынки для российского металла, вынуждая «Норникель» и других экспортеров перенаправлять потоки в Азию, преимущественно в Китай, зачастую с дисконтом.
По данным таможенной статистики, в марте поставки палладия в Гонконг выросли на 34% по сравнению с февралём.
Аналитики отмечают, что перенастройка логистики займёт от шести до девяти месяцев, и в этот период возможны временные ценовые скачки.
Минпромторг РФ заявил, что ищет новые рынки сбыта, включая Индию и Бразилию.
Несмотря на трудности, «Норникель» подтвердил годовой прогноз производства платиноидов на уровне 2,5 млн унций.
"""

In [15]:
articles = {
        "Apple": article_apple,
        "Авторынок": article_autos,
        "Палладий": article_palladium
}

for title, text in articles.items():
        print(f"СТАТЬЯ: {title}")
        print(f"Длина текста (символов): {len(text)}")

        summary_sum = score_sentences(text, use_average=False, top_k=4)
        print("\n--- Аннотация (СУММА частот, 4 предложения) ---")
        for i, sent in enumerate(summary_sum, 1):
            print(f"{i}. {sent}")

        summary_avg = score_sentences(text, use_average=True)
        print("\n--- Аннотация (СРЕДНЯЯ частота на слово, 1 предложение) ---")
        print(summary_avg[0])

СТАТЬЯ: Apple
Длина текста (символов): 699

--- Аннотация (СУММА частот, 4 предложения) ---
1. 20 апреля Тим Кук заявил, что покинет пост генерального директора и станет исполнительным председателем совета директоров компании.
2. Как сообщает AFP, генеральный директор Тим Кук назвал этот период «лучшим мартовским кварталом за всю историю».
3. Apple отчиталась о рекордном спросе на линейку iPhone 17, а рекордные продажи цифровых услуг помогли технологическому гиганту превзойти ожидания аналитиков.
4. Его преемником назначен Джон Тернус, который сейчас занимает пост вице-президента по аппаратной инженерии.

--- Аннотация (СРЕДНЯЯ частота на слово, 1 предложение) ---
20 апреля Тим Кук заявил, что покинет пост генерального директора и станет исполнительным председателем совета директоров компании.
СТАТЬЯ: Авторынок
Длина текста (символов): 4398

--- Аннотация (СУММА частот, 4 предложения) ---
1. Во многом речь идет не о появлении принципиально новых шоурумов, вторит директор департамента п